In [1]:
!echo 'Welcome to simple Souffle graph benchmarking.'
!echo 'Behold the power of The Machine:'
!echo "CPUs: $(nproc) / RAM: $(free -h | awk '/^Mem:/ {print $2}')"

Welcome to simple Souffle graph benchmarking.
Behold the power of The Machine:
CPUs: 32 / RAM: 125Gi


In [2]:
from logica.common import benchmarking

In [3]:
test_items = ['a', 'b', 'c']

In [4]:
%%loop ("Test Problem", test_items)
print("{loop_parameter}")

Running a.
a
Running b.
b
Running c.
c
 === Timing for Test Problem ===
+---------+------------------------+
| problem | time                   |
+---------+------------------------+
| a       | 0.00022841000100015663 |
| b       | 0.00019157000133418478 |
| c       | 0.00020624000171665102 |
+---------+------------------------+


In [5]:
print(benchmarking.reports[0])

 === Timing for Test Problem ===
+---------+------------------------+
| problem | time                   |
+---------+------------------------+
| a       | 0.00022841000100015663 |
| b       | 0.00019157000133418478 |
| c       | 0.00020624000171665102 |
+---------+------------------------+


In [6]:
from logica import colab_logica

Could not import google.cloud.bigquery.
Could not import google.cloud.auth.
Could not import google.colab.widgets.


In [7]:
%%logica G,S

@AttachDatabase("db", "graphs.db");

N() = 100;
D() = 10;

GStep1(a, b, i:) distinct :-
  a = NaturalHash("a-" ++ ToString(i)) % N(),
  b = NaturalHash("b-" ++ ToString(i)) % N(),
  a != b,
  i in Range(ToInt64(N() * D() * 1.1));

I() = l[N() * D()] :-
  l = Array{ i -> i :- GStep1(i:) };

G(a, b) :-
  GStep1(a, b, i:),
  i < I();


@Ground(G5c, "db.G5c", copy_to_file: "data/g5c.csv");
G5c := G(N: 500);
@Ground(G6c, "db.G6c", copy_to_file: "data/g6c.csv");
G6c := G(N: 600);
@Ground(G7c, "db.G7c", copy_to_file: "data/g7c.csv");
G7c := G(N: 700);

@Ground(G1k, "db.G1k", copy_to_file: "data/g1k.csv");
G1k := G(N: 1000);
@Ground(G2k, "db.G2k", copy_to_file: "data/g2k.csv");
G2k := G(N: 2000);
@Ground(G3k, "db.G3k", copy_to_file: "data/g3k.csv");
G3k := G(N: 3000);
@Ground(G4k, "db.G4k", copy_to_file: "data/g4k.csv");
G4k := G(N: 4000);
@Ground(G5k, "db.G5k", copy_to_file: "data/g5k.csv");
G5k := G(N: 5000);

S() += 1 :- G5c() | G6c() | G7c() | G1k() | G2k() | G3k() | G4k() | G5k();
 


Query is stored at G_sql variable.
The following table is stored at G variable.


,col0,col1
0,15,6
1,10,76
2,77,42
3,0,23
4,84,67
5,77,34
6,46,55
7,96,49
8,75,85
9,21,27


Query is stored at S_sql variable.
The following table is stored at S variable.


,logica_value
0,168000


In [8]:
graphs = ['G1k', 'G2k', 'G3k', 'G4k', 'G5k']

In [9]:
%%logica Tree, NumNodes

@AttachDatabase("db", "graphs.db");

Depth() = 5;
DegreeRange(1, 5);

InnerTree(x, y) :-
    (x = "0" | InnerTree(something, x)),
    Length(x) < Depth(),
    DegreeRange(d1, d2),
    d = NaturalHash("d-" ++ x) % (d2 - d1) + d1,
    i in Range(d),
    y = x ++ ToString(i);

# Creating a non-recursive face, so we can customly ground it.
Tree(x, y) :- InnerTree(x, y);

@Ground(Tree5, "db.Tree5", copy_to_file: "data/tree5.csv");
Tree5 := Tree(Depth: 5);
@Ground(Tree6, "db.Tree6", copy_to_file: "data/tree6.csv");
Tree6 := Tree(Depth: 6);
@Ground(Tree7, "db.Tree7", copy_to_file: "data/tree7.csv");
Tree7 := Tree(Depth: 7);
@Ground(Tree8, "db.Tree8", copy_to_file: "data/tree8.csv");
Tree8 := Tree(Depth: 8);
@Ground(Tree9, "db.Tree9", copy_to_file: "data/tree9.csv");
Tree9 := Tree(Depth: 9);
@Ground(Tree10, "db.Tree10", copy_to_file: "data/tree10.csv");
Tree10 := Tree(Depth: 10);
@Ground(Tree11, "db.Tree11", copy_to_file: "data/tree11.csv");
Tree11 := Tree(Depth: 11);
@Ground(Tree12, "db.Tree12", copy_to_file: "data/tree12.csv");
Tree12 := Tree(Depth: 12);

NumNodes(tree5: Sum{ 1 :- Tree5()},
         tree6: Sum{ 1 :- Tree6()},
         tree7: Sum{ 1 :- Tree7()},
         tree8: Sum{ 1 :- Tree8()},
         tree9: Sum{ 1 :- Tree9()},
         tree10: Sum{ 1 :- Tree10()},
         tree11: Sum{ 1 :- Tree11()},
         tree12: Sum{ 1 :- Tree12()}
        );


Query is stored at Tree_sql variable.
The following table is stored at Tree variable.


,col0,col1
0,0,00
1,0,01
2,00,000
3,000,0000
4,000,0001
5,000,0002
6,000,0003
7,0001,00010
8,0001,00011
9,0001,00012


Query is stored at NumNodes_sql variable.
The following table is stored at NumNodes variable.


,tree5,tree6,tree7,tree8,tree9,tree10,tree11,tree12
0,37,85,206,505,1257,3152,7793,19456


In [10]:
trees = ['Tree7', 'Tree8', 'Tree9', 'Tree10', 'Tree11', 'Tree12']

In [11]:
%%loop ("Transitive Closure", graphs)
%%logica TC, TCSize

@AttachDatabase("db", "graphs.db");

G(a, b) :- db.{loop_parameter}(a, b);

@Recursive(TC, ∞, stop: Done);
TC(a, b) distinct :- G(a, b);
TC(a, c) distinct :- TC(a, b), G(b, c);

PrevTC(a, b) :- TC(a, b);

Done() :- TC(a, b) => PrevTC(a, b);

TCSize() += 1 :- TC();

Running G1k.
Query is stored at TC_sql variable.
The following table is stored at TC variable.


,col0,col1
0,403,288
1,150,58
2,192,467
3,592,505
4,236,257
5,811,623
6,753,82
7,911,322
8,236,9
9,991,698


Query is stored at TCSize_sql variable.
The following table is stored at TCSize variable.


,logica_value
0,1000000


 
 
Running G2k.
Query is stored at TC_sql variable.
The following table is stored at TC variable.


...

 
 
Running G5k.
Query is stored at TC_sql variable.
The following table is stored at TC variable.


,col0,col1
0,2444,3295
1,2825,3878
2,2291,4755
3,1441,2485
4,3232,3402
5,68,2112
6,1465,4165
7,3771,2751
8,2650,2449
9,4657,2284


Query is stored at TCSize_sql variable.
The following table is stored at TCSize variable.


,logica_value
0,24995000


 
 
 === Timing for Transitive Closure ===
+---------+--------------------+
| problem | time               |
+---------+--------------------+
| G1k     | 1.7456023790000472 |
| G2k     | 3.2657229699980235 |
| G3k     | 5.309176804999879  |
| G4k     | 8.26903399799994   |
| G5k     | 12.238729950000561 |
+---------+--------------------+


In [12]:
%%loop ("Pairwise Distances", graphs)
%%logica D, DSize

@AttachDatabase("db", "graphs.db");

G(a, b) :- db.{loop_parameter}(a, b);

@Recursive(D, ∞, stop: Done);
D(a, b) Min= 1 :- G(a, b);
D(a, c) Min= D(a, b) + 1 :- G(b, c);

PrevD(a, b) :- D(a, b);

Done() :- D(a, b) => PrevD(a, b);

DSize() += 1 :- D();

Running G1k.
Query is stored at D_sql variable.
The following table is stored at D variable.


,col0,col1,logica_value
0,327,192,1
1,212,365,1
2,629,433,1
3,102,358,1
4,947,212,1
5,826,777,1
6,683,738,1
7,694,958,1
8,187,454,1
9,95,6,1


Query is stored at DSize_sql variable.
The following table is stored at DSize variable.


,logica_value
0,1000000


 
 
Running G2k.
Query is stored at D_sql variable.
The following table is stored at D variable.


...

 
 
Running G5k.
Query is stored at D_sql variable.
The following table is stored at D variable.


,col0,col1,logica_value
0,2924,4006,1
1,3076,2764,1
2,3997,2957,1
3,3686,2237,1
4,1581,77,1
5,4895,458,1
6,3374,2950,1
7,738,470,1
8,2123,2622,1
9,863,594,1


Query is stored at DSize_sql variable.
The following table is stored at DSize variable.


,logica_value
0,24995000


 
 
 === Timing for Pairwise Distances ===
+---------+--------------------+
| problem | time               |
+---------+--------------------+
| G1k     | 2.270142572997429  |
| G2k     | 3.8740923719997227 |
| G3k     | 6.949681735000922  |
| G4k     | 10.555769121001504 |
| G5k     | 15.13741886400021  |
+---------+--------------------+


In [13]:
from logica.common import graph
graph.InstallRequire()

In [14]:

graph.DirectedGraph(Tree)

In [20]:
%%loop ("Same Generation", trees)
%%logica SGCheck, SGGold


@AttachDatabase("db", "graphs.db");

Tree(x, y) :- db.{loop_parameter}(x, y);
#Tree(x, y) :- db.Tree12(x, y);

Node(x) distinct :- x in [a, b], Tree(a, b), Str(x);

# Tree("a", "b");
# Tree("a", "c");
# Tree("c", "d");

@Recursive(SG, ∞, stop: Done);
SG(x, y) distinct :- Tree(a, x), Tree(a, y);
SG(x, y) distinct :- SG(a, b), Tree(a, x), Tree(b, y);
PrevSG(x, y) :- SG(x, y);
Done() :- Sum{ 1 :- SG(x, y) } == Sum{ 1 :- PrevSG(x, y) };
    
SGCheck() += 1 :- SG();
SGGold() += 1 :- Node(a), Node(b), Length(a) == Length(b);


Running Tree7.
Query is stored at SGCheck_sql variable.
The following table is stored at SGCheck variable.


,logica_value
0,17506


Query is stored at SGGold_sql variable.
The following table is stored at SGGold variable.


,logica_value
0,17507


 
 
Running Tree8.
Query is stored at SGCheck_sql variable.
The following table is stored at SGCheck variable.


...

 
 
Running Tree12.
Query is stored at SGCheck_sql variable.
The following table is stored at SGCheck variable.


,logica_value
0,161827886


Query is stored at SGGold_sql variable.
The following table is stored at SGGold variable.


,logica_value
0,161827887


 
 
 === Timing for Same Generation ===
+---------+--------------------+
| problem | time               |
+---------+--------------------+
| Tree7   | 1.4477802029978193 |
| Tree8   | 1.3823529130022507 |
| Tree9   | 1.5115272420007386 |
| Tree10  | 1.9040136679977877 |
| Tree11  | 4.701103225001134  |
| Tree12  | 20.051564998000686 |
+---------+--------------------+


In [16]:
benchmarking.timing

{'a': 0.00022841000100015663,
 'b': 0.00019157000133418478,
 'c': 0.00020624000171665102,
 'G1k': 2.270142572997429,
 'G2k': 3.8740923719997227,
 'G3k': 6.949681735000922,
 'G4k': 10.555769121001504,
 'G5k': 15.13741886400021,
 'Tree7': 1.2040877750005166,
 'Tree8': 0.9793701880007575,
 'Tree9': 1.5164861519988335,
 'Tree10': 1.8636865869993926,
 'Tree11': 4.7414349719983875,
 'Tree12': 19.595385920998524}

In [17]:
%%loop ("Same Generation Logica-style", trees)
%%logica SGCheck, SGGold


@AttachDatabase("db", "graphs.db");

Tree(x, y) :- db.{loop_parameter}(x, y);
#Tree(x, y) :- db.Tree7(x, y);

Node(x) distinct :- x in [a, b], Tree(a, b), Str(x);
# Tree("a", "b");
# Tree("a", "c");
# Tree("c", "d");


@Recursive(Gen, ∞, stop: Done);
Gen(x) Min= x :- Node(x);
NextGen(Gen(x)) Min= Gen(y) :- Tree(x, y);
Gen(y) Min= NextGen(Gen(x)) :- Tree(x, y);

PrevGen(x) = Gen(x);
Done() :- Gen(x) => Gen(x) = PrevGen(x);

SG(x, y) :- Gen(x) = Gen(y);
    

SGCheck() += 1 :- SG();
SGGold() += 1 :- Node(a), Node(b), Length(a) == Length(b);


Running Tree7.
Query is stored at SGCheck_sql variable.
The following table is stored at SGCheck variable.


,logica_value
0,17507


Query is stored at SGGold_sql variable.
The following table is stored at SGGold variable.


,logica_value
0,17507


 
 
Running Tree8.
Query is stored at SGCheck_sql variable.
The following table is stored at SGCheck variable.


...

 
 
Running Tree12.
Query is stored at SGCheck_sql variable.
The following table is stored at SGCheck variable.


,logica_value
0,161827887


Query is stored at SGGold_sql variable.
The following table is stored at SGGold variable.


,logica_value
0,161827887


 
 
 === Timing for Same Generation Logica-style ===
+---------+--------------------+
| problem | time               |
+---------+--------------------+
| Tree7   | 1.1043042769997555 |
| Tree8   | 1.0176126980004483 |
| Tree9   | 1.0705868459990597 |
| Tree10  | 1.1368336560008174 |
| Tree11  | 1.2924413349974202 |
| Tree12  | 2.0945264440015308 |
+---------+--------------------+


In [18]:
benchmarking.timing

{'a': 0.00022841000100015663,
 'b': 0.00019157000133418478,
 'c': 0.00020624000171665102,
 'G1k': 2.270142572997429,
 'G2k': 3.8740923719997227,
 'G3k': 6.949681735000922,
 'G4k': 10.555769121001504,
 'G5k': 15.13741886400021,
 'Tree7': 1.1043042769997555,
 'Tree8': 1.0176126980004483,
 'Tree9': 1.0705868459990597,
 'Tree10': 1.1368336560008174,
 'Tree11': 1.2924413349974202,
 'Tree12': 2.0945264440015308}

In [19]:
for report in benchmarking.reports:
    print(report)

 === Timing for Test Problem ===
+---------+------------------------+
| problem | time                   |
+---------+------------------------+
| a       | 0.00022841000100015663 |
| b       | 0.00019157000133418478 |
| c       | 0.00020624000171665102 |
+---------+------------------------+
 === Timing for Transitive Closure ===
+---------+--------------------+
| problem | time               |
+---------+--------------------+
| G1k     | 1.7456023790000472 |
| G2k     | 3.2657229699980235 |
| G3k     | 5.309176804999879  |
| G4k     | 8.26903399799994   |
| G5k     | 12.238729950000561 |
+---------+--------------------+
 === Timing for Pairwise Distances ===
+---------+--------------------+
| problem | time               |
+---------+--------------------+
| G1k     | 2.270142572997429  |
| G2k     | 3.8740923719997227 |
| G3k     | 6.949681735000922  |
| G4k     | 10.555769121001504 |
| G5k     | 15.13741886400021  |
+---------+--------------------+
 === Timing for Same Generation ===
